# Mabhas Compliance — RAG Pipeline (Steps 1 & 2)
### End-to-end: corpus upload → embedding → pgvector ingestion → retrieval test

**Run cells top to bottom. Each cell prints its own status.**  
Connection uses keyword arguments — no URL parsing, no `@` confusion.

| Step | What happens | Time |
|------|-------------|------|
| 0 | Install libraries | ~2 min |
| 1 | Upload `mabhas_clauses.json` | seconds |
| 2 | Configure and test database connection | seconds |
| 3 | Apply schema | seconds |
| 4 | Load embedding model | ~5 min first run |
| 5 | Filter corpus | seconds |
| 6 | Embed + ingest | ~8 min |
| 7 | Retrieval test | seconds |
| 8 | Final summary report | seconds |

## Step 0 — Install dependencies

In [ ]:
!pip install -q sentence-transformers psycopg2-binary pgvector
print('✓ Libraries installed')

## Step 1 — Upload your corpus
Upload `mabhas_clauses.json` from your downloaded zip.  
Or use the Drive option if the file is already there.

In [ ]:
# Option A — upload directly from your computer
from google.colab import files as colab_files
import json, pathlib

print('Select mabhas_clauses.json ...')
uploaded = colab_files.upload()

CORPUS_PATH = None
for name in uploaded:
    pathlib.Path(name).write_bytes(uploaded[name])
    CORPUS_PATH = name
    print(f'✓ Uploaded: {name}  ({len(uploaded[name]):,} bytes)')

corpus = json.loads(pathlib.Path(CORPUS_PATH).read_text(encoding='utf-8'))
print(f'✓ Parsed {len(corpus)} clauses')

In [ ]:
# Option B — load from Google Drive (run INSTEAD of Option A if file is on Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# import json
# CORPUS_PATH = '/content/drive/MyDrive/mabhas_clauses.json'  # adjust path
# corpus = json.load(open(CORPUS_PATH, encoding='utf-8'))
# print(f'✓ Loaded {len(corpus)} clauses from Drive')

## Step 2 — Configure and test database connection

**Fill in the four fields below, then run the cell.**  
The password prompt is a hidden input — type it and press Enter.  

> ⚠️ **Do NOT paste `user@host` into the host field.**  
> Host must be only the hostname, e.g. `xxxx.db.arvandbaas.ir`  
> User and password go in their own fields.

In [ ]:
# @title Database configuration { display-mode: 'form' }
import getpass, psycopg2
from pgvector.psycopg2 import register_vector

#  ↓ Fill these four fields ↓
DB_HOST = 'xxxx.db.arvandbaas.ir'   # @param {type:'string'}   hostname ONLY
DB_PORT = '5432'                     # @param {type:'string'}
DB_NAME = 'compliance'               # @param {type:'string'}
DB_USER = 'postgres'                 # @param {type:'string'}

DB_PASSWORD = getpass.getpass('Password (hidden): ')

# ── connection helper used by every cell below ──────────────────────────
# Uses keyword arguments — completely avoids URL-parsing problems with
# special characters (@, #, %, etc.) in passwords or hostnames.
def get_conn():
    conn = psycopg2.connect(
        host     = DB_HOST,
        port     = int(DB_PORT),
        dbname   = DB_NAME,
        user     = DB_USER,
        password = DB_PASSWORD,
    )
    register_vector(conn)
    return conn

# ── connection test ──────────────────────────────────────────────────────
print(f'Testing connection to {DB_HOST}:{DB_PORT}/{DB_NAME} as {DB_USER} ...')
try:
    conn = get_conn()
    with conn.cursor() as cur:
        cur.execute('SELECT version();')
        ver = cur.fetchone()[0]
    conn.close()
    print(f'✓ Connected!  PostgreSQL version: {ver[:40]}')
except psycopg2.OperationalError as e:
    print(f'✗ Connection failed:\n  {e}')
    print()
    print('Common causes:')
    print('  • Wrong host — must be hostname only, e.g. xxxx.db.arvandbaas.ir')
    print('  • Wrong username or password')
    print('  • Database does not exist yet (create it first in your panel)')
    print('  • Firewall blocking port 5432 from Colab IPs')

## Step 3 — Apply schema
Creates the `mabhas_clauses` table and indexes.  
Safe to re-run — uses `IF NOT EXISTS`.

In [ ]:
SCHEMA_SQL = '''
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS mabhas_clauses (
    id                       SERIAL PRIMARY KEY,
    mabhas_part              TEXT          NOT NULL,
    article_id               TEXT          NOT NULL,
    heading_fa               TEXT,
    text_fa                  TEXT          NOT NULL,
    text_en                  TEXT,
    rule_type                TEXT,
    entities                 JSONB,
    applicable_occupancies   JSONB,
    applicable_height_groups JSONB,
    embedding                vector(1024)  NOT NULL,
    created_at               TIMESTAMPTZ   DEFAULT now(),
    CONSTRAINT uq_part_article UNIQUE (mabhas_part, article_id)
);
CREATE INDEX IF NOT EXISTS idx_mabhas_embedding
    ON mabhas_clauses USING hnsw (embedding vector_cosine_ops);
CREATE INDEX IF NOT EXISTS idx_mabhas_rule_type ON mabhas_clauses (rule_type);
CREATE INDEX IF NOT EXISTS idx_mabhas_part      ON mabhas_clauses (mabhas_part);
CREATE INDEX IF NOT EXISTS idx_mabhas_occ
    ON mabhas_clauses USING gin (applicable_occupancies);
'''

conn = get_conn()
with conn.cursor() as cur:
    cur.execute(SCHEMA_SQL)
conn.commit()
conn.close()
print('✓ Schema applied — table and indexes ready')

## Step 4 — Load embedding model
Downloads `intfloat/multilingual-e5-large` (~2 GB) on first run.  
Subsequent runs use the Colab cache and take only seconds.

In [ ]:
from sentence_transformers import SentenceTransformer
import threading

EMBEDDING_DIM = 1024
MODEL_NAME    = 'intfloat/multilingual-e5-large'
DEFAULT_SCOPE = {'M-4', 'all_residential', 'any'}

_model = None
_lock  = threading.Lock()

def get_model():
    global _model
    if _model is None:
        with _lock:
            if _model is None:
                print(f'Downloading {MODEL_NAME} ...')
                _model = SentenceTransformer(MODEL_NAME)
                print('✓ Model loaded')
    return _model

def embed_passages(texts, batch_size=16):
    if not texts: return []
    prefixed = [f'passage: {t}' for t in texts]
    vecs = get_model().encode(prefixed, batch_size=batch_size,
                              normalize_embeddings=True, show_progress_bar=True)
    return [v.tolist() for v in vecs]

def embed_query(text):
    v = get_model().encode([f'query: {text}'], normalize_embeddings=True,
                           show_progress_bar=False)
    return v[0].tolist()

get_model()
print(f'✓ Embedding dimension: {EMBEDDING_DIM}')

## Step 5 — Filter corpus to scope and preview

In [ ]:
from collections import Counter

def is_in_scope(clause, scope):
    if clause.get('skip_category'): return False
    occ = clause.get('applicable_occupancies')
    if occ is not None: return bool(set(occ) & scope)
    v1 = clause.get('applicable_to_floor_plan')
    return bool(v1) if v1 is not None else True

def build_passage(clause):
    parts = []
    if clause.get('heading_fa'): parts.append(clause['heading_fa'].strip())
    parts.append(clause['text_fa'].strip())
    if clause.get('text_en'):    parts.append(clause['text_en'].strip())
    return '\n'.join(p for p in parts if p)

keep = [c for c in corpus if is_in_scope(c, DEFAULT_SCOPE)]
skip = [c for c in corpus if not is_in_scope(c, DEFAULT_SCOPE)]

print(f'Total clauses : {len(corpus)}')
print(f'Will ingest   : {len(keep)}')
print(f'Will skip     : {len(skip)}')
print()
print('Kept by rule_type:')
for rt, n in Counter(c.get('rule_type') for c in keep).most_common():
    print(f'  {str(rt):12}  {n}')
print()
print('Skipped by category:')
for cat, n in Counter(c.get('skip_category') for c in skip
                       if c.get('skip_category')).most_common():
    print(f'  {cat:22}  {n}')

## Step 6 — Embed and ingest
Embeds the in-scope clauses and upserts into the database.  
Idempotent — safe to re-run.

In [ ]:
from psycopg2.extras import Json, execute_values

BATCH = 32

print(f'Embedding {len(keep)} clauses in batches of {BATCH} ...')
passages   = [build_passage(c) for c in keep]
embeddings = embed_passages(passages, batch_size=BATCH)
print(f'✓ {len(embeddings)} embeddings computed')

rows = []
for clause, emb in zip(keep, embeddings):
    rows.append((
        str(clause['mabhas_part']),
        str(clause['article_id']),
        clause.get('heading_fa'),
        clause['text_fa'],
        clause.get('text_en'),
        clause.get('rule_type'),
        Json(clause['entities']) if clause.get('entities') is not None else None,
        Json(clause.get('applicable_occupancies') or []),
        Json(clause.get('applicable_height_groups') or []),
        emb,
    ))

UPSERT = '''
    INSERT INTO mabhas_clauses
        (mabhas_part, article_id, heading_fa, text_fa,
         text_en, rule_type, entities,
         applicable_occupancies, applicable_height_groups, embedding)
    VALUES %s
    ON CONFLICT (mabhas_part, article_id) DO UPDATE SET
        heading_fa               = EXCLUDED.heading_fa,
        text_fa                  = EXCLUDED.text_fa,
        text_en                  = EXCLUDED.text_en,
        rule_type                = EXCLUDED.rule_type,
        entities                 = EXCLUDED.entities,
        applicable_occupancies   = EXCLUDED.applicable_occupancies,
        applicable_height_groups = EXCLUDED.applicable_height_groups,
        embedding                = EXCLUDED.embedding;
'''

conn = get_conn()
with conn.cursor() as cur:
    execute_values(cur, UPSERT, rows)
conn.commit()
with conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM mabhas_clauses;')
    total_in_db = cur.fetchone()[0]
conn.close()

print(f'✓ Upserted {len(rows)} rows')
print(f'✓ Total rows now in database: {total_in_db}')

## Step 7 — Retrieval test
Five queries against the index — confirms the pipeline is working end to end.

In [ ]:
from psycopg2.extras import RealDictCursor

def retrieve(query, top_k=3, rule_type=None, mabhas_part=None):
    qvec  = embed_query(query)
    sql   = ['SELECT mabhas_part, article_id, rule_type, text_en,',
             '       1 - (embedding <=> %s) AS score',
             'FROM mabhas_clauses WHERE TRUE']
    params = [qvec]
    if rule_type:
        sql.append('AND rule_type = %s'); params.append(rule_type)
    if mabhas_part:
        sql.append('AND mabhas_part = %s'); params.append(mabhas_part)
    sql.append('ORDER BY embedding <=> %s LIMIT %s')
    params += [qvec, top_k]
    conn = get_conn()
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute(' '.join(sql), params)
        rows = [dict(r) for r in cur.fetchall()]
    conn.close()
    for r in rows: r['score'] = round(float(r['score']), 3)
    return rows

TESTS = [
    ('minimum bedroom area',               'numeric'),
    ('kitchen must not open into bathroom', 'spatial'),
    ('minimum ceiling height habitable room','numeric'),
    ('egress route safe exit access',       'spatial'),
    ('stair tread riser width',             'numeric'),
]

print('=' * 70)
for query, rt in TESTS:
    hits = retrieve(query, top_k=1, rule_type=rt)
    if hits:
        h = hits[0]
        print(f'Query : {query}')
        print(f'  Hit : [{h["article_id"]}]  score={h["score"]}  type={h["rule_type"]}')
        en = (h['text_en'] or '')[:110]
        print(f'  Text: {en}...')
    else:
        print(f'Query: {query}  ->  NO RESULT (check ingestion)')
    print('-' * 70)

## Step 8 — Final summary report

In [ ]:
from psycopg2.extras import RealDictCursor
from IPython.display import display, HTML

conn = get_conn()

with conn.cursor(cursor_factory=RealDictCursor) as cur:
    cur.execute('''
        SELECT rule_type,
               COUNT(*) AS total,
               COUNT(CASE WHEN applicable_occupancies @> ''["M-4"]''::jsonb THEN 1 END) AS m4_specific,
               COUNT(CASE WHEN applicable_occupancies @> ''["all_residential"]''::jsonb THEN 1 END) AS all_res
        FROM mabhas_clauses
        GROUP BY rule_type ORDER BY total DESC;
    ''')
    rows = cur.fetchall()

with conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM mabhas_clauses;')
    grand_total = cur.fetchone()[0]
conn.close()

html = [
    '<table style="border-collapse:collapse;font-family:monospace;font-size:14px">',
    '<tr style="background:#2b5797;color:white">',
    '<th style="padding:8px 16px">rule_type</th>',
    '<th style="padding:8px 16px">total</th>',
    '<th style="padding:8px 16px">M-4 specific</th>',
    '<th style="padding:8px 16px">all_residential</th></tr>',
]
colors = {'numeric':'#e8f4e8','spatial':'#e8eef8',
          'definition':'#fdf6e3','exception':'#fdeaea'}
for r in rows:
    bg = colors.get(r['rule_type'],'#fff')
    html.append(f'<tr style="background:{bg}">'
        f'<td style="padding:6px 16px;font-weight:bold">{r["rule_type"]}</td>'
        f'<td style="padding:6px 16px;text-align:center">{r["total"]}</td>'
        f'<td style="padding:6px 16px;text-align:center">{r["m4_specific"]}</td>'
        f'<td style="padding:6px 16px;text-align:center">{r["all_res"]}</td></tr>')
html.append(
    f'<tr style="background:#333;color:white;font-weight:bold">'
    f'<td style="padding:8px 16px">TOTAL</td>'
    f'<td style="padding:8px 16px;text-align:center">{grand_total}</td>'
    f'<td colspan=2></td></tr></table>')
display(HTML(''.join(html)))
print()
print('✓ Steps 1 + 2 complete.')
print(f'  {grand_total} Mabhas clauses are searchable in your database.')
print('  Ready for Step 3: spatial_graph.py')